# The F1 Score — Why the Harmonic Mean

Precision and recall pull against each other. Predict the positive class more freely and recall
rises while precision falls; predict it more sparingly and the reverse. A single number that
summarizes both has to refuse to let either one be sacrificed — and the ordinary average does not
refuse. A model with precision 1.0 and recall 0.0 averages to a respectable 0.5 while being
completely useless.

The **F1 score** uses the *harmonic* mean instead, and this notebook shows geometrically why that
choice is the right one.

## Learning objectives

- Compute F1 as the harmonic mean of precision and recall
- Show that F1 always lies between the two, and always nearer the smaller
- Explain why the harmonic mean punishes an imbalance that the arithmetic mean tolerates
- Read the F1 surface over the precision–recall square

## Background

This notebook assumes precision and recall as definitions:

$$\text{precision} = \frac{TP}{TP + FP}, \qquad \text{recall} = \frac{TP}{TP + FN}$$

Precision asks "of everything I flagged, how much was right?"; recall asks "of everything I should
have flagged, how much did I catch?". Neither is computed from data here — the notebook treats them
as two free numbers in $[0,1]$ and studies the function that combines them.

**Prerequisites:** none. `U1-6_Classify-4_ParamSweep` and `U1_Diabetes-3_Classification` use F1 on
real predictions.

**Dataset:** none — precision and recall are sampled or swept directly.

**References:** https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. One random pair

The F1 score is the harmonic mean of precision $p$ and recall $r$:

$$F_1 = \frac{2pr}{p + r} = \frac{2}{\frac{1}{p} + \frac{1}{r}}$$

The second form is the definition of a harmonic mean — the reciprocal of the average of the
reciprocals — and it is where the behaviour comes from. A reciprocal magnifies small numbers, so a
small $p$ or $r$ dominates the sum $\frac{1}{p} + \frac{1}{r}$ and drags the result down.

Two properties follow, and the plot below shows both on one randomly drawn pair:

- $F_1$ always lies **between** $p$ and $r$.
- It sits **nearer the smaller** of the two, not midway. Where the arithmetic mean would sit exactly
  in the middle, the harmonic mean leans toward the weaker score.

Re-run the cell a few times. The green marker never escapes the interval, and it consistently
crowds the lower of the two.

It is worth being concrete about why the arithmetic mean fails here. Take a model that flags exactly
one patient, and is right: precision is 1.0, recall is near 0. The arithmetic mean reports 0.5 — a
middling score for a model that found almost nothing. The harmonic mean reports approximately 0,
which is the truthful summary.

The same asymmetry runs the other way. A model that flags *everyone* has recall 1.0 and precision
equal to the base rate. Averaging rewards it; the harmonic mean does not. Any single-number summary
of two quantities has to decide what to do when they diverge, and $F_1$ decides in favour of the
pessimist.


In [ ]:
p = np.random.rand()
r = np.random.rand()
f = 2 * p * r / ( p + r )

print(f"p: {p}")
print(f"r: {r}")
print(f"f: {f}")

plt.plot( p, 0, 'r.', markersize=10, label='Precision')
plt.plot( r, 0, 'b.', markersize=10, label='Recall')
plt.plot( f, 0, 'g.', markersize=10, label='F1')
plt.xlim(0,1)
plt.legend()
plt.show()

## 2. The whole surface

One pair at a time only goes so far. Here $p$ and $r$ are swept across $[0,1]^2$ and three surfaces
are drawn together: the plane $z = p$, the plane $z = r$, and the F1 surface itself.

What to look for:

- Along the **diagonal** $p = r$, all three surfaces meet — with nothing to penalize, the harmonic
  mean equals both inputs.
- Away from the diagonal, the F1 surface dips **below both planes**, and the further apart $p$ and
  $r$ are, the deeper the dip.
- Along the **edges** where either $p \to 0$ or $r \to 0$, F1 collapses to 0 regardless of how large
  the other is. This is the property that makes F1 usable on imbalanced problems: a model that
  ignores the minority class cannot hide behind its precision.

The `1e-9` in the denominator guards the $p = r = 0$ corner, where the formula is otherwise $0/0$.

> **A note on `%matplotlib qt`.** This cell switches to an interactive backend so the surface can be
> rotated with the mouse — worth doing, since the dip below the two planes is much easier to see
> from an angle. It opens a separate window rather than plotting inline, and it needs a desktop
> session. Swap in the commented `%matplotlib inline` for a static figure inside the notebook.

One more property visible in the surface: $F_1$ is **symmetric** in $p$ and $r$ — swapping them
leaves the value unchanged, so the surface is a mirror image across the diagonal. That symmetry is
an assumption, not a law. It says a false positive and a false negative cost the same, which is
false for most real problems and is exactly what the reward matrix in
`U1-7_Imbalance-4_ClassWeights` exists to correct. The generalized $F_\beta$ score,

$$F_\beta = (1+\beta^2)\,\frac{p \cdot r}{\beta^2 p + r}$$

breaks the symmetry deliberately: $\beta > 1$ weights recall more heavily, $\beta < 1$ favours
precision, and $\beta = 1$ recovers the surface plotted here.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Create a range of values from 0 to 1
pre = np.linspace(0, 1, 100)
rec = np.linspace(0, 1, 100)

# Create a meshgrid
PRE, REC = np.meshgrid(pre, rec)

# Calculate F1 score
F1 = 2 * PRE * REC / (PRE + REC + 1e-9)

%matplotlib qt        # interactive 3-D window - rotate the surface with the mouse
#%matplotlib inline   # static inline figure instead

# Create a 3D plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Plot the surface
ax.plot_surface(PRE, REC, PRE, color='r', alpha=0.3)
ax.plot_surface(PRE, REC, REC, color='b', alpha=0.3)
ax.plot_surface(PRE, REC, F1,  cmap='nipy_spectral')

# Labeling
ax.set_xlabel('Precision (PRE)')
ax.set_ylabel('Recall (REC)')
ax.set_zlabel('F1 Score')
ax.set_title('3D Plot of F1 Score vs Precision and Recall')
#plt.show()

## 3. Review

- **$F_1 = \dfrac{2pr}{p+r}$ is the harmonic mean** of precision and recall — the reciprocal of the
  average of the reciprocals.
- **It always lies between $p$ and $r$, and always nearer the smaller.** Reciprocals magnify small
  values, so the weaker score dominates.
- **It equals both inputs only on the diagonal $p = r$**, and dips further below them the more
  unbalanced the pair is.
- **If either goes to zero, $F_1$ goes to zero** no matter the other. That is precisely why $F_1$ is
  the metric of choice for rare-event problems, where accuracy and even the arithmetic mean let a
  model that never predicts the minority class look acceptable.